Chatbot And RAG Evaluation
Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

How to create test datasets
How to run your RAG application on those datasets
How to measure your application's performance using different evaluation metrics
Overview
A typical RAG evaluation workflow consists of three main steps:

Creating a dataset with questions and their expected answers
Running your RAG application on those questions
Using evaluators to measure how well your application performed, looking at factors like:
Answer relevance
Answer accuracy
Retrieval quality
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

In [17]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [7]:
from langsmith import Client

client = Client()

dataset_name = "Chatbot3 Evaluation"
dataset = client.create_dataset(dataset_name)
dataset = client.create_examples (
    dataset_id = dataset.id,
    examples = [{
        "inputs":{"question":"What is LangChain?"},
        "outputs":{"answer":"A framework for building LLM applications"}
    },{
        "inputs":{"question":"What is LangSmith?"},
        "outputs":{"answer":"A platform for observing and evaluating LLM applications"}
    },{
        "inputs":{"question":"What is OpenAI?"},
        "outputs":{"answer":"A company that creates Large Language Models"}
    },{
        "inputs":{"question":"What is Google?"},
        "outputs":{"answer":"A technology company known for search"}
    },{
        "inputs":{"question":"What is Mistral?"},
        "outputs":{"answer":"A company that creates Large Language Models"}
    }]

)



In [8]:
dataset

{'example_ids': ['39cef569-275c-49fe-8090-6415bdc74109',
  '1457f99d-36ae-4426-858b-e8874d164c89',
  '95734fc1-8148-43f9-8210-7bb311d8be8d',
  '41ed3b77-e32a-47e6-9afe-afdfd2e5b412',
  'cdd216d0-413a-449e-8330-a5ddb52cb462'],
 'count': 5,
 'as_of': '2026-07-02T16:20:24.284179915Z'}

Define Metrics (LLM As A Judge)

In [18]:
# LLM as a judge
import openai
from langsmith import wrappers

In [19]:
openai_client = wrappers.wrap_openai(openai.OpenAI())

eval_instructions = "You are an expert professor specialized in grading students answers to questions."

def correctness(inputs:dict, outputs:dict, reference_outputs:dict)->bool:
    user_content = f"""
    You are grading the following question:
    {inputs['question']}
    
    Here is the real answer:
    {reference_outputs['answer']}
    
    You are grading the following predicted answer:
    {outputs['response']}
    
    Respond with CORRECT or INCORRECT:
    Grade:
    """

    response = openai_client.chat.completions.create(
        model = "gpt-4o-mini",
        temperature=0,
        messages=[
            {"role":"system", "content":eval_instructions},
            {"role":"user", "content":user_content}
        ]
    ).choices[0].message.content

    return response == "CORRECT"

In [20]:
## Concisions- checks whether the actual output is less than 2x the length of expected result

def concisions(outputs:dict, reference_outputs:dict)->bool:
    return int(len(outputs['response']) < 2 * len(reference_outputs["answer"]))

In [21]:
## Run Evaluation

default_instrcutions = "Respond to the users question in short, concise manner(one short sentence)."
def my_app(question:str,model:str="gpt-4o-mini",instructions:str=default_instrcutions) -> str:
    return openai_client.chat.completions.create(
        model = model,
        temperature = 0,
        messages = [{
            "role":"user","content":question},
            {"role":"system","content":instructions
        }]
    ).choices[0].message.content

In [22]:
## Call my_app for every data points
def ls_target(inputs:str)->dict:
    return {"response":my_app(inputs["question"])}

In [23]:
# Run our evaluation
experiment_result = client.evaluate(
    ls_target, # Your AI system
    data = dataset_name,
    evaluators=[correctness,concisions],
    experiment_prefix="openai-4o-mini-chatbot"
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-ff328fd6' at:
https://smith.langchain.com/o/b607ec67-52db-4d83-875a-de6b8b226f33/datasets/04c2ae30-809a-4bcc-9bfb-c07882516849/compare?selectedSessions=19b487b8-01e3-4731-89e6-bc369efa7b64




3it [00:11,  3.86s/it]


## For gpt-4-turbo

In [19]:
## Run Evaluation

default_instrcutions = "Respond to the users question in short, concise manner(one short sentence)."
def my_app(question:str,model:str="gpt-4-turbo",instructions:str=default_instrcutions) -> str:
    return openai_client.chat.completions.create(
        model = model,
        temperature = 0,
        messages = [{
            "role":"user","content":question},
            {"role":"system","content":instructions
        }]
    ).choices[0].message.content

In [20]:
## Call my_app for every data points
def ls_target(inputs:str)->dict:
    return {"response":my_app(inputs["question"])}

In [21]:
# Run our evaluation
experiment_result = client.evaluate(
    ls_target, # Your AI system
    data = dataset_name,
    evaluators=[correctness,concisions],
    experiment_prefix="openai-4-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-4-turbo-chatbot-bdb9c6f3' at:
https://smith.langchain.com/o/b607ec67-52db-4d83-875a-de6b8b226f33/datasets/15242166-dd66-413e-b3f6-8989d76352cf/compare?selectedSessions=1ec1056d-c085-4dc8-a2b1-ec5dc96ec61f




5it [00:19,  3.96s/it]


In [1]:
## Evaluation for RAG

In [24]:
# Libraries
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/z9/f1bl4swj6g31nwl_qk7b4_xm0000gp/T/ipykernel_15061/1190681243.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [25]:
# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

In [26]:
# Load documents from web url
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for subList in docs for item in subList]

In [27]:
# Initialize the text splitter
from langchain_core.tools import retriever


text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size =250, chunk_overlap =0)

# Split the document into chunks 
splitted_docs = text_splitter.split_documents(docs_list)

# Add the embeddings and splitting into vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents = splitted_docs,
    embedding = OpenAIEmbeddings()
)

# Retriever
retriever = vectorstore.as_retriever(k=6)

In [28]:
retriever.invoke("What is agents?")

[Document(id='4d3d1c38-a532-47f0-a5ec-a1bf57bed093', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [19]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model = "openai:gpt-4o-mini")
# llm

In [29]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("openai:gpt-4o-mini")
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x12a853c80>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x12f2a4f50>, root_client=<openai.OpenAI object at 0x12a8e6330

In [30]:
from langsmith import traceable

# Add decorator

@traceable()
def rag_bot(question:str) -> dict:
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.      
    Use three sentences maximum and keep the answer concise.
    Documents: {docs_string}"""

    ## llm invoke 
    ai_msg = llm.invoke([
        {"role":"user","content":question},
        {"role":"system","content":instructions}
    ])
    return {"answer":ai_msg.content,"documents":docs}


In [31]:
rag_bot("What is agents?")

{'answer': '"Agents" refers to autonomous entities that utilize large language models (LLMs) to make decisions and interact within various environments. These agents can include generative agents that simulate human-like behavior and LLM-powered autonomous systems that incorporate planning, memory, and reflection for task management. They are designed to perform complex tasks by breaking them down into smaller subgoals and learning from past experiences.',
 'documents': [Document(id='4d3d1c38-a532-47f0-a5ec-a1bf57bed093', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful g

In [33]:
# Datasets
from langsmith import Client

client = Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

# Create dataset and examples in LangSmith
dataset_name = "rag test evaluation3"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id = dataset.id,
    examples = examples
)

{'example_ids': ['4e596c9a-09fb-4b9c-88ff-de040f87cc3c',
  'd6b87532-0501-412f-8d04-860daf1363e9',
  '75e7cd4f-55fb-4840-aea2-c8aea6150646'],
 'count': 3,
 'as_of': '2026-07-06T13:57:51.870037937Z'}

In [4]:
# Evaluators or Metrics
# Correctness: Response vs reference answer
# Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
# Mode: Requires a ground truth (reference) answer supplied through a dataset
# Evaluator: Use LLM-as-judge to assess answer correctness.

In [34]:
from typing_extensions import Annotated, TypedDict

## Correctness/Output schema
class Correctness(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str,...,"Explain your reasoning for the score"]
    correct: Annotated[bool,...,"True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_openai import ChatOpenAI

grader_llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0).with_structured_output(Correctness,method = "json_schema", strict=True)

## Evaluator
def correctness_eval(inputs:dict, outputs:dict, reference_outputs:dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
        QUESTION: {inputs['question']}
        GROUND TRUTH ANSWER:{reference_outputs['answer']}
        STUDENT ANSWER: {outputs['answer']}"""

        # Run evaluator
    grade = grader_llm.invoke([
            {"role":"user","content": answers},
            {"role":"system","content":correctness_instructions}
        ])
    return grade['correct']

In [35]:
#Relevance: Response vs input
#The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

In [36]:
#Groundedness: Response vs retrieved docs
#Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

In [37]:
# Retrieval Relevance: Retrieved docs vs input

# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

In [38]:
# Run evaluation
from sys import version


def target(inputs:dict)->dict:
    return rag_bot(inputs['question'])

experiment_result = client.evaluate(
    target,
    data = dataset_name,
    evaluators=[correctness_eval, groundedness, relevance, retrieval_relevance],
    experiment_prefix = "rag-doc-relevance2",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)

experiment_result.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance2-5bec226d' at:
https://smith.langchain.com/o/b607ec67-52db-4d83-875a-de6b8b226f33/datasets/8b4082db-54aa-4fde-8b8c-fe4b932eb74d/compare?selectedSessions=39bdddeb-65bf-4245-8e0b-7905c784c0d2




3it [00:33, 11.16s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness_eval,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection by integr...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",True,False,True,False,2.661709,4e596c9a-09fb-4b9c-88ff-de040f87cc3c,019f37b8-ee1a-7c60-a568-9a1cdb846962
1,What are five types of adversarial attacks?,Five types of adversarial attacks are:\n\n1. *...,[page_content='Black-box attacks assume that a...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,3.008528,75e7cd4f-55fb-4840-aea2-c8aea6150646,019f37b9-2081-7a93-be82-28d878ca8758
2,What are the types of biases that can arise wi...,The types of biases that can arise with few-sh...,[page_content='Zero-shot and few-shot learning...,None,The biases that can arise with few-shot prompt...,True,True,True,True,2.080931,d6b87532-0501-412f-8d04-860daf1363e9,019f37b9-484f-7060-9aca-0910ca222394
